# LPSN - List of Prokaryotic names with Standing in Nomenclature

The **List of Prokaryotic names with Standing in Nomenclature**
([LPSN](https://lpsn.dsmz.de/)) is the authoritative reference catalogue of
validly published names for **Bacteria** and **Archaea**. It is maintained by
the Leibniz Institute DSMZ (German Collection of Microorganisms and Cell
Cultures) and tracks:

- Every validly published genus, species, and subspecies name (with its
  original publication and any later reclassifications)
- Nomenclatural status (correct name, synonym, illegitimate, etc.)
- Type strains and their deposits in culture collections
- Higher taxonomy (domain, phylum, class, order, family)

LPSN is a designated **ELIXIR Core Data Resource** because taxonomy underpins
essentially every downstream microbiology workflow - from 16S rRNA
classification to clinical pathogen reporting.

**Access routes:**

1. **REST API** at `https://api.lpsn.dsmz.de/` - rich per-record access but
   requires a free registration and an `Authorization` header.
2. **Bulk CSV download** at <https://lpsn.dsmz.de/downloads> - a single
   flat-file snapshot (`lpsn_gss.csv`) containing every genus/species/subspecies
   record. No authentication is required, which makes it the right choice for
   a reproducible demonstration notebook.

This notebook uses the **CSV bulk download** so anyone can re-run it without
registering for API credentials.

In [ ]:
"""Imports for the LPSN ingestion notebook."""

import time
from pathlib import Path

import polars as pl
import requests

# Project-local data directory (created on first run).
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

## TODO

**Ingest data**
- [x] Identify and document the bulk CSV download endpoint
- [x] Download `lpsn_gss.csv` with on-disk caching
- [x] Parse into a Polars DataFrame with correct dtypes
- [x] Summarise taxonomy level counts (genera / species / subspecies)

**Explore and clean**
- [ ] Audit missingness per column (type strain, authors, publication year)
- [ ] Normalise nomenclatural status categories
- [ ] Split binomial names into genus / species epithet columns
- [ ] Flag orphan records (species with no resolvable genus entry)

**Analysis**
- [ ] Growth of validly published names per year (bacteria vs. archaea)
- [ ] Genus size distribution (species per genus) - heavy-tailed?
- [ ] Synonym graph: basonym -> current name chains
- [ ] Cross-link type strains to BacDive / StrainInfo identifiers

**Visualization**
- [ ] Cumulative name count over time, faceted by domain
- [ ] Log-log rank/frequency plot of genus sizes (Zipf-style)
- [ ] Sunburst of domain -> phylum -> class coverage
- [ ] Heatmap of publication year x nomenclatural status

**Statistical analysis**
- [ ] Fit a power-law to species-per-genus and compare to log-normal (AIC)
- [ ] Test whether description rates differ by domain (Poisson regression)
- [ ] Bootstrap CIs for year-on-year growth with multiple-testing correction
- [ ] Survival analysis of names (time from publication to reclassification)

## 1. Ingest Data

### 1.1 Download strategy

The LPSN downloads page (<https://lpsn.dsmz.de/downloads>) exposes a single
comprehensive flat file, the **Genus/Species/Subspecies table** (`lpsn_gss.csv`).
Every row is one nomenclatural record and columns include:

| Column                    | Description                                           |
|---------------------------|-------------------------------------------------------|
| `genus_name`              | Genus component of the name                           |
| `sp_epithet`              | Species epithet (blank for genus-only rows)           |
| `subsp_epithet`           | Subspecies epithet (blank unless rank = subspecies)   |
| `status`                  | Nomenclatural status (e.g. `correct name, validly published under the ICNP`) |
| `authors`                 | Taxonomic authorship string                           |
| `nomenclatural_type`      | Type strain / type species pointer                    |
| `record_no`               | Stable LPSN record identifier                         |
| `record_lnk`              | Canonical URL on lpsn.dsmz.de                         |

We fetch the file once, cache it under `data/`, and re-use the cache on every
subsequent run so the notebook is cheap to re-execute and works offline.

> **Note on the API.** The richer per-record JSON API at
> `https://api.lpsn.dsmz.de/` requires registration and a Bearer token, which
> would force every reader of this notebook to create an account. The flat CSV
> contains all of the taxonomy-level information we need for the demos below.

### 1.2 Cached download helper

In [ ]:
# Known bulk-download endpoints for LPSN, in preference order.
# The DSMZ publishes the full Genus/Species/Subspecies flat file at both the
# API host and the main lpsn.dsmz.de host. Multiple candidates give us a
# resilient download even if one CDN path changes.
LPSN_CSV_CANDIDATES = (
    "https://api.lpsn.dsmz.de/download/lpsn_gss_latest.csv",
    "https://lpsn.dsmz.de/downloads/lpsn_gss.csv",
)

# Local cache location. The file is a few tens of MB, well worth caching.
LPSN_CSV_PATH = DATA_DIR / "lpsn_gss.csv"


def download_lpsn_csv(
    dest: Path = LPSN_CSV_PATH,
    urls: tuple[str, ...] = LPSN_CSV_CANDIDATES,
    force: bool = False,
    timeout: int = 60,
) -> Path:
    """Download the LPSN Genus/Species/Subspecies bulk CSV with on-disk caching.

    Parameters
    ----------
    dest : pathlib.Path, optional
        Destination path for the cached file. Defaults to ``data/lpsn_gss.csv``.
    urls : tuple of str, optional
        Ordered list of candidate download URLs. The first URL that returns
        an HTTP 200 response is used; the rest act as fallbacks.
    force : bool, optional
        If ``True``, re-download even when a cached copy already exists.
    timeout : int, optional
        Per-request timeout in seconds.

    Returns
    -------
    pathlib.Path
        Path to the cached CSV file on disk.

    Raises
    ------
    RuntimeError
        If every candidate URL fails.

    Notes
    -----
    The file is streamed to disk in 1 MiB chunks to keep memory usage flat
    regardless of release size.
    """
    if dest.exists() and not force:
        size_mb = dest.stat().st_size / 1024**2
        print(f"Using cached {dest} ({size_mb:.1f} MB)")
        return dest

    last_error: Exception | None = None
    for url in urls:
        try:
            print(f"Downloading {url} ...")
            t0 = time.perf_counter()
            with requests.get(url, stream=True, timeout=timeout) as response:
                response.raise_for_status()
                # Write to a temp sibling file, then atomically rename - this
                # prevents a half-written file from poisoning the cache.
                tmp = dest.with_suffix(dest.suffix + ".part")
                with tmp.open("wb") as fh:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            fh.write(chunk)
                tmp.replace(dest)
            elapsed = time.perf_counter() - t0
            size_mb = dest.stat().st_size / 1024**2
            print(f"Saved {dest} ({size_mb:.1f} MB in {elapsed:.1f}s)")
            return dest
        except requests.RequestException as exc:
            print(f"  failed ({exc!s}); trying next candidate")
            last_error = exc

    raise RuntimeError(
        f"Could not download LPSN CSV from any candidate URL: {urls}"
    ) from last_error


csv_path = download_lpsn_csv()
csv_path

### 1.3 Parse into a Polars DataFrame

We load the CSV with an **explicit schema**. LPSN ships integer-ish columns
(`record_no`, `record_lnk_no`, `genus_record_no`, ...) that `read_csv`'s
inference will sometimes collapse to `Int64` and sometimes to `Utf8` depending
on the release - pinning the types up front keeps downstream joins stable.

We then derive a single `rank` column from the presence/absence of
`sp_epithet` and `subsp_epithet` so every row has a clean categorical
taxonomic level.

In [ ]:
# LPSN ships these as strings even when they look numeric (blanks are common),
# so we read them as Utf8 and cast the unambiguous integer columns afterwards.
LPSN_STRING_COLUMNS = [
    "genus_name",
    "sp_epithet",
    "subsp_epithet",
    "status",
    "authors",
    "nomenclatural_type",
    "record_lnk",
]
LPSN_INT_COLUMNS = [
    "record_no",
    "record_lnk_no",
    "genus_record_no",
    "sp_record_no",
]


def load_lpsn(path: Path = LPSN_CSV_PATH) -> pl.DataFrame:
    """Load the cached LPSN bulk CSV into a typed Polars DataFrame.

    Parameters
    ----------
    path : pathlib.Path, optional
        Path to the cached ``lpsn_gss.csv`` file.

    Returns
    -------
    polars.DataFrame
        One row per nomenclatural record with an added ``rank`` column whose
        values are ``"genus"``, ``"species"``, or ``"subspecies"``.

    Notes
    -----
    Integer columns that are absent in a given LPSN release are silently
    skipped - LPSN occasionally adds or renames cross-reference columns and
    we do not want schema drift to break ingestion.
    """
    df = pl.read_csv(
        path,
        infer_schema_length=0,  # treat everything as Utf8 first for safety
        ignore_errors=True,
        null_values=["", "NA", "null"],
    )

    # Cast integer-looking cross-references where they are present.
    int_casts = [
        pl.col(c).cast(pl.Int64, strict=False).alias(c)
        for c in LPSN_INT_COLUMNS
        if c in df.columns
    ]
    if int_casts:
        df = df.with_columns(int_casts)

    # Derive a clean taxonomic rank column from the name components. Rows with
    # a subspecies epithet are subspecies; those with only a species epithet
    # are species; otherwise the row names a genus.
    df = df.with_columns(
        pl.when(pl.col("subsp_epithet").is_not_null())
        .then(pl.lit("subspecies"))
        .when(pl.col("sp_epithet").is_not_null())
        .then(pl.lit("species"))
        .otherwise(pl.lit("genus"))
        .alias("rank")
    )

    return df


lpsn = load_lpsn()
print(f"Loaded {lpsn.height:,} LPSN records x {lpsn.width} columns")
lpsn.head()

### 1.4 Summarise taxonomy levels

Quick sanity-check: how many genera, species, and subspecies does this release
contain, and what are the most speciose genera?

In [ ]:
def summarise_ranks(df: pl.DataFrame) -> pl.DataFrame:
    """Count LPSN records at each taxonomic rank.

    Parameters
    ----------
    df : polars.DataFrame
        Output of :func:`load_lpsn` (must contain a ``rank`` column).

    Returns
    -------
    polars.DataFrame
        Two columns: ``rank`` and ``n_records``, sorted from most to least
        populous.
    """
    return (
        df.group_by("rank")
        .agg(pl.len().alias("n_records"))
        .sort("n_records", descending=True)
    )


rank_summary = summarise_ranks(lpsn)
print("Records per rank:")
print(rank_summary)

# Species-per-genus distribution: one of the more scientifically interesting
# slices of LPSN - it typically follows a heavy-tailed distribution driven by
# a handful of huge genera (e.g. Streptomyces, Pseudomonas).
species_per_genus = (
    lpsn.filter(pl.col("rank") == "species")
    .group_by("genus_name")
    .agg(pl.len().alias("n_species"))
    .sort("n_species", descending=True)
)

print(f"\nDistinct genera (with >=1 species): {species_per_genus.height:,}")
print("Top 10 most speciose genera:")
print(species_per_genus.head(10))

### 1.5 Persist a typed snapshot

We write a Parquet snapshot next to the raw CSV. Parquet is a columnar format
with per-column compression and preserves our Polars dtypes exactly, so the
exploration and analysis sections can reload the data in milliseconds.

In [ ]:
LPSN_PARQUET_PATH = DATA_DIR / "lpsn_gss.parquet"

# zstd gives ~3-4x better compression than snappy on taxonomic text columns
# at a negligible read cost for a file this size.
lpsn.write_parquet(LPSN_PARQUET_PATH, compression="zstd")

parquet_mb = LPSN_PARQUET_PATH.stat().st_size / 1024**2
csv_mb = LPSN_CSV_PATH.stat().st_size / 1024**2
print(
    f"Wrote {LPSN_PARQUET_PATH} ({parquet_mb:.1f} MB) "
    f"from {csv_mb:.1f} MB source CSV "
    f"({csv_mb / parquet_mb:.1f}x compression)"
)